In [ ]:
import google.generativeai as genai
import json
import time
import re

# ==================== CẤU HÌNH API ====================
genai.configure(api_key='')
model = genai.GenerativeModel('gemini-2.5-flash-lite')
# ======================================================

# Prompt template cho 5 context - GIỮ NGUYÊN
quiz_generation_prompt_5 = """
Bạn là chuyên gia lịch sử Việt Nam. Hãy tạo 5 câu hỏi trắc nghiệm về lịch sử Việt Nam dựa trên 5 đoạn thông tin lịch sử dưới đây.

YÊU CẦU QUAN TRỌNG:
- Mỗi câu hỏi PHẢI dựa trực tiếp vào thông tin trong một đoạn tương ứng
- Câu hỏi phải HOÀN TOÀN TỰ NHIÊN, KHÔNG được đề cập "theo văn bản", "đoạn văn", "ngữ liệu" hay bất kỳ tham chiếu nào đến nguồn
- Giải thích cũng phải TỰ NHIÊN, KHÔNG được dùng "văn bản cho biết", "theo tài liệu" - hãy trình bày như một sự thật lịch sử
- Mỗi câu hỏi có 4 đáp án (A, B, C, D) với 3 đáp án sai hợp lý và 1 đáp án đúng
- **QUAN TRỌNG: Các đáp án PHẢI LIÊN QUAN TRỰC TIẾP đến câu hỏi và nội dung đoạn thông tin**
- **ĐÁP ÁN CHO MỖI CÂU HỎI PHẢI KHÁC NHAU và PHÙ HỢP với ngữ cảnh cụ thể của từng câu**
- **KHÔNG ĐƯỢC sử dụng cùng một bộ đáp án cho nhiều câu hỏi khác nhau**
- Đáp án sai phải có vẻ hợp lý, không được quá vô lý hoặc hoàn toàn không liên quan
- Đảm bảo đáp án đúng là chính xác theo thông tin

THÔNG TIN LỊCH SỬ 1:
{context1}

THÔNG TIN LỊCH SỬ 2:
{context2}

THÔNG TIN LỊCH SỬ 3:
{context3}

THÔNG TIN LỊCH SỬ 4:
{context4}

THÔNG TIN LỊCH SỬ 5:
{context5}

ĐỊNH DẠNG ĐẦU RA BẮT BUỘC - PHẢI THEO ĐÚNG FORMAT NÀY:

Câu hỏi 1: [câu hỏi tự nhiên về thông tin 1]
A. [đáp án A cụ thể cho câu hỏi 1]
B. [đáp án B cụ thể cho câu hỏi 1] 
C. [đáp án C cụ thể cho câu hỏi 1]
D. [đáp án D cụ thể cho câu hỏi 1]
Đáp án đúng 1: [chỉ ghi A, B, C hoặc D]
Giải thích 1: [giải thích tự nhiên, trình bày như sự thật]

Câu hỏi 2: [câu hỏi tự nhiên về thông tin 2]
A. [đáp án A cụ thể cho câu hỏi 2]
B. [đáp án B cụ thể cho câu hỏi 2] 
C. [đáp án C cụ thể cho câu hỏi 2]
D. [đáp án D cụ thể cho câu hỏi 2]
Đáp án đúng 2: [chỉ ghi A, B, C hoặc D]
Giải thích 2: [giải thích tự nhiên, trình bày như sự thật]

Câu hỏi 3: [câu hỏi tự nhiên về thông tin 3]
A. [đáp án A cụ thể cho câu hỏi 3]
B. [đáp án B cụ thể cho câu hỏi 3] 
C. [đáp án C cụ thể cho câu hỏi 3]
D. [đáp án D cụ thể cho câu hỏi 3]
Đáp án đúng 3: [chỉ ghi A, B, C hoặc D]
Giải thích 3: [giải thích tự nhiên, trình bày như sự thật]

Câu hỏi 4: [câu hỏi tự nhiên về thông tin 4]
A. [đáp án A cụ thể cho câu hỏi 4]
B. [đáp án B cụ thể cho câu hỏi 4] 
C. [đáp án C cụ thể cho câu hỏi 4]
D. [đáp án D cụ thể cho câu hỏi 4]
Đáp án đúng 4: [chỉ ghi A, B, C hoặc D]
Giải thích 4: [giải thích tự nhiên, trình bày như sự thật]

Câu hỏi 5: [câu hỏi tự nhiên về thông tin 5]
A. [đáp án A cụ thể cho câu hỏi 5]
B. [đáp án B cụ thể cho câu hỏi 5] 
C. [đáp án C cụ thể cho câu hỏi 5]
D. [đáp án D cụ thể cho câu hỏi 5]
Đáp án đúng 5: [chỉ ghi A, B, C hoặc D]
Giải thích 5: [giải thích tự nhiên, trình bày như sự thật]
"""

def generate_5_quizzes_from_contexts(contexts):
    """Generate 5 quizzes từ 5 contexts sử dụng Gemini API"""
    if len(contexts) != 5:
        raise ValueError("Cần chính xác 5 contexts")
    
    prompt = quiz_generation_prompt_5.format(
        context1=contexts[0],
        context2=contexts[1], 
        context3=contexts[2],
        context4=contexts[3],
        context5=contexts[4]
    )
    
    try:
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.9,
                top_p=0.9,
                top_k=50,
                max_output_tokens=2000,
            )
        )
        return response.text
    except Exception as e:
        print(f" Lỗi API: {e}")
        return None

def parse_5_quizzes_response(response_text):
    """Phân tích kết quả từ Gemini thành list of 5 quiz data - ĐÃ SỬA LỖI PARSE"""
    if not response_text:
        return None
        
    try:
        # Tách response thành các phần riêng cho từng câu hỏi
        quizzes = []
        
        # Tìm tất cả các câu hỏi bằng regex
        question_blocks = re.split(r'Câu hỏi \d+:', response_text)
        # Bỏ qua phần đầu tiên vì nó là phần trước câu hỏi 1
        question_blocks = question_blocks[1:]
        
        if len(question_blocks) < 5:
            print(f" Chỉ tìm thấy {len(question_blocks)}/5 câu hỏi")
            return None
        
        for i, block in enumerate(question_blocks[:5]):  # Chỉ xử lý 5 câu đầu
            quiz_data = {}
            q_num = i + 1
            
            # Tách block thành các dòng
            lines = [line.strip() for line in block.split('\n') if line.strip()]
            
            # Dòng đầu tiên là câu hỏi
            if lines:
                quiz_data['question'] = lines[0]
            
            # Tìm các đáp án trong block này
            options_found = 0
            current_option = None
            
            for line in lines[1:]:  # Bỏ qua dòng câu hỏi đầu tiên
                # Kiểm tra xem dòng có bắt đầu bằng A., B., C., D. không
                option_match = re.match(r'^([A-D])\.\s*(.+)', line)
                if option_match:
                    current_option = option_match.group(1)
                    quiz_data[current_option] = option_match.group(2).strip()
                    options_found += 1
                # Kiểm tra đáp án đúng
                elif line.startswith(f'Đáp án đúng {q_num}:'):
                    correct_match = re.search(r'Đáp án đúng \d+:\s*([A-D])', line)
                    if correct_match:
                        quiz_data['correct'] = correct_match.group(1)
                # Kiểm tra giải thích
                elif line.startswith(f'Giải thích {q_num}:'):
                    explanation_match = re.search(r'Giải thích \d+:\s*(.+)', line)
                    if explanation_match:
                        quiz_data['explanation'] = explanation_match.group(1).strip()
            
            # Kiểm tra xem có đủ thông tin không
            required_fields = ['question', 'A', 'B', 'C', 'D', 'correct', 'explanation']
            if all(field in quiz_data for field in required_fields):
                quizzes.append(quiz_data)
            else:
                print(f" Câu hỏi {q_num} thiếu thông tin: {[f for f in required_fields if f not in quiz_data]}")
                return None
        
        return quizzes if len(quizzes) == 5 else None
            
    except Exception as e:
        print(f" Lỗi parse 5 quizzes: {e}")
        import traceback
        print(f"Traceback: {traceback.format_exc()}")
        return None

def create_training_example(context, generated_quiz):
    """Tạo một mẫu training theo format chat"""
    
    user_message = f"Hãy tạo một câu hỏi trắc nghiệm với 4 lựa chọn (3 sai, 1 đúng) và giải thích đáp án dựa trên thông tin sau:\n\n{context}"
    
    assistant_message = f"**Câu hỏi:** {generated_quiz['question']}\n\n**Đáp án:**\nA. {generated_quiz['A']}\nB. {generated_quiz['B']}\nC. {generated_quiz['C']}\nD. {generated_quiz['D']}\n\n**Đáp án đúng:** {generated_quiz['correct']}\n\n**Giải thích:** {generated_quiz['explanation']}"
    
    return {
        "messages": [
            {
                "role": "system", 
                "content": "Bạn là chuyên gia lịch sử Việt Nam. Hãy tạo câu hỏi trắc nghiệm dựa trên ngữ liệu lịch sử được cung cấp."
            },
            {
                "role": "user", 
                "content": user_message
            },
            {
                "role": "assistant", 
                "content": assistant_message
            }
        ]
    }

def log_quiz_group(group_index, contexts, quizzes):
    """Log chi tiết một nhóm quiz để kiểm tra chất lượng"""
    print(f"\n{'='*80}")
    print(f" LOG NHÓM {group_index} - KIỂM TRA CHẤT LƯỢNG (5 CÂU HỎI)")
    print(f"{'='*80}")
    
    for i, (context, quiz) in enumerate(zip(contexts, quizzes)):
        print(f"\n CÂU HỎI {i+1}:")
        print(f" Context {i+1} (tóm tắt): {context[:80]}...")
        print(f" Câu hỏi: {quiz['question']}")
        print(f" Đáp án:")
        print(f"   A. {quiz['A']}")
        print(f"   B. {quiz['B']}")
        print(f"   C. {quiz['C']}")
        print(f"   D. {quiz['D']}")
        print(f" Đáp án đúng: {quiz['correct']}")
        print(f" Giải thích: {quiz['explanation']}")
        print(f"{'-'*60}")
    
    print(f"✅ NHÓM {group_index} - ĐÃ KIỂM TRA XONG 5 CÂU HỎI")
    print(f"{'='*80}\n")

def read_rag_data(file_path):
    """Đọc dữ liệu RAG từ file JSONL"""
    data = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    item = json.loads(line.strip())
                    data.append(item)
        print(f" Đã đọc {len(data)} mẫu từ {file_path}")
        return data
    except Exception as e:
        print(f" Lỗi đọc file: {e}")
        return []

def generate_training_data(input_file, output_file, num_samples=2000):
    """Generate training data từ file JSONL input"""
    
    rag_data = read_rag_data(input_file)
    if not rag_data:
        return
    
    rag_data = rag_data[:num_samples]
    training_examples = []
    success_count = 0
    failed_count = 0
    
    print(f" Bắt đầu generate {len(rag_data)} câu hỏi...")
    
    # Nhóm 5 context một
    grouped_data = [rag_data[i:i+5] for i in range(0, len(rag_data), 5)]
    
    for i, group in enumerate(grouped_data):
        if len(group) < 5:
            print(f" Nhóm cuối chỉ có {len(group)} context, bỏ qua")
            failed_count += len(group)
            continue
        
        contexts = [item.get('text', '') for item in group]
        if any(not context for context in contexts):
            print(f"❌ Nhóm {i+1}: Có context rỗng")
            failed_count += 5
            continue
        
        print(f" Đang xử lý nhóm {i+1}/{len(grouped_data)} (context {i*5+1}-{i*5+5})...")
        
        quiz_response = generate_5_quizzes_from_contexts(contexts)
        if quiz_response:
            print(f" Response từ Gemini (200 ký tự đầu): {quiz_response[:200]}...")
            parsed_quizzes = parse_5_quizzes_response(quiz_response)
            if parsed_quizzes:
                # LOG CHI TIẾT để kiểm tra chất lượng
                log_quiz_group(i + 1, contexts, parsed_quizzes)
                
                for j, quiz in enumerate(parsed_quizzes):
                    training_example = create_training_example(contexts[j], quiz)
                    training_examples.append(training_example)
                    success_count += 1
                    
                    with open(output_file, 'a', encoding='utf-8') as f:
                        f.write(json.dumps(training_example, ensure_ascii=False) + '\n')
                
                print(f" Nhóm {i+1} - Thành công: 5/5 - Tổng: {success_count}")
            else:
                print(f" Parse thất bại nhóm {i+1}")
                print(f" Response gốc từ Gemini (500 ký tự đầu):")
                print(quiz_response[:500] + "..." if len(quiz_response) > 500 else quiz_response)
                failed_count += 5
        else:
            print(f" Generate thất bại nhóm {i+1}")
            failed_count += 5
        
        time.sleep(5)
        
        if (i + 1) % 1 == 0:  # Log mỗi nhóm để debug
            print(f"📊 Tiến độ: {i+1}/{len(grouped_data)} nhóm - Thành công: {success_count}, Thất bại: {failed_count}")
    
    print(f"🎉 Hoàn thành! Tổng: {success_count} thành công, {failed_count} thất bại")
    return training_examples

# ==================== CHƯƠNG TRÌNH CHÍNH ====================
if __name__ == "__main__":
    input_file = "/kaggle/input/datausedtogenquiz/jsonl_1.jsonl"  # File JSONL chứa context
    output_file = "/kaggle/working/quiz_training_data.jsonl"  # File output
    
    print(" Bắt đầu generate quiz training data...")
    print(f" Input: {input_file}")
    print(f" Output: {output_file}")
    print(" 5 câu hỏi/request - ĐÃ SỬA LỖI PARSE ĐÁP ÁN")
    print("-" * 50)
    
    # Tạo file output trống
    with open(output_file, 'w', encoding='utf-8') as f:
        pass
    
    # Generate dữ liệu
    start_time = time.time()
    training_data = generate_training_data(input_file, output_file, 2000)
    end_time = time.time()
    
    print("-" * 50)
    print(f" Đã hoàn thành trong {((end_time - start_time)/60):.1f} phút!")
    print(f" Dữ liệu được lưu tại: {output_file}")
    print(f" Số lượng mẫu training: {len(training_data) if training_data else 0}")